In [ ]:
# pip install umap-learn
# pip install umap
import os, sys
import time
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import torch
import umap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from rdkit.Chem import AllChem as Chem
from sklearn.preprocessing import StandardScaler
import random
import yaml
random.seed(16)

In [ ]:
def get_fp(list_of_smi):
    fingerprints = []
    mols = [Chem.MolFromSmiles(x) for x in list_of_smi]
    idx_to_remove = []
    for idx,mol in enumerate(mols):
        try:
            fprint = Chem.GetMorganFingerprintAsBitVect(mol, 2, useFeatures=False)
            fingerprints.append(fprint)
        except:
            idx_to_remove.append(idx)
    
    smi_to_keep = [smi for i,smi in enumerate(list_of_smi) if i not in idx_to_remove]
    return fingerprints, smi_to_keep

def get_embedding(data):
    data_scaled = StandardScaler().fit_transform(data)
    
    embedding = umap.UMAP(n_neighbors=10,
                          min_dist=0.5,
                          metric='correlation',
                          random_state=16).fit_transform(data_scaled)
    
    return embedding

In [ ]:
def draw_umap(embedding, lim_origin):

    fig, ax = plt.subplots(figsize=(14, 10))

    plt.xlim([np.min(embedding[:,0])-0.5, np.max(embedding[:,0])+1.5])
    plt.ylim([np.min(embedding[:,1])-0.5, np.max(embedding[:,1])+0.5])

    labelsize = 20
    plt.xlabel('UMAP1', fontsize=labelsize, fontproperties="SimHei") 
    plt.ylabel('UMAP2', fontsize=labelsize, fontproperties="SimHei") 

    # Hide the right and top spines
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    plt.scatter(embedding[:lim_origin, 0], embedding[:lim_origin, 1], 
                lw=0, c='#8b8d94', label='Real Molecule', alpha=0.85, s=180,
                marker="o") # c1c0c4 9d9ea1 8b8d94 6c6f78
    
    plt.scatter(embedding[lim_origin:, 0], embedding[lim_origin:, 1], 
                lw=0, c='#0d0be9', label='Generate Molecule', alpha=1.0, s=180, 
                marker="^") # , edgecolors='k', linewidth=1
    
    leg = plt.legend(prop={'size': labelsize}, loc='upper right', markerscale=1.50, scatteryoffsets=[0.5, 0.5, 0.5], frameon=False, labelspacing=1.00, handlelength=0.5)
    leg.get_frame().set_alpha(0.9)
    leg.get_frame().set_edgecolor('white')
    
    plt.setp(ax, xticks=[], yticks=[])
    plt.savefig("MolecularChemicalSpatialDistribution.png", dpi=300)
    plt.show()

In [ ]:
start = time.time()
random.seed(1234)
train_path='../data_crossdocked/final_filter_train.yaml'

with open(train_path, 'r') as f:
    train_smi = list(yaml.full_load(f).values())

gen3_path='../save/pre/crossdocked/char/hgnn/2025_01_05_20/sample_300_30_True_1_1_1741635931/metrics_each.csv'
df_g3 = pd.read_csv(gen3_path)
smiles_generate = df_g3['smiles'].tolist()



smiles_origin = random.sample(train_smi, 1000)
smiles_generate = random.sample(smiles_generate, 1000)


origin_fp, origin_smiles = get_fp(smiles_origin)

origin_fp = np.array(origin_fp)
generate_fp, generate_smiles = get_fp(smiles_generate)
generate_fp = np.array(generate_fp)
# print(origin_fp.shape)
# print(generate_fp.shape)


all_data = np.concatenate([origin_fp, generate_fp], axis=0)
# print(all_data.shape)
embedding = get_embedding(all_data)
# print(embedding.shape)
# print(embedding)
lim_origin = origin_fp.shape[0]
draw_umap(embedding, lim_origin)
end = time.time()
print(f'UMAP PROJECTION DONE in {end - start:.04} seconds')

In [ ]:
start = time.time()
random.seed(1234)
origin_data = pd.read_csv("protein_ligand.csv") # , sep="\t"
smiles_origin = origin_data["SMILES"].tolist()

smiles_generate = []
generate_data = pd.read_csv("MMF2Drug_all.csv")
gen_data = generate_data["SMILES"]
for smiles in gen_data:
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None and len(smiles) > 20:
        smiles_generate.append(smiles)
        
smiles_origin = random.sample(smiles_origin, 1000)
smiles_generate = random.sample(smiles_generate, 1000)
origin_fp, origin_smiles = get_fp(smiles_origin)

origin_fp = np.array(origin_fp)
generate_fp, generate_smiles = get_fp(smiles_generate)
generate_fp = np.array(generate_fp)
# print(origin_fp.shape)
# print(generate_fp.shape)
all_data = np.concatenate([origin_fp, generate_fp], axis=0)
# print(all_data.shape)
embedding = get_embedding(all_data)
# print(embedding.shape)
# print(embedding)

lim_origin = origin_fp.shape[0]
draw_umap(embedding, lim_origin)
end = time.time()
print(f'UMAP PROJECTION DONE in {end - start:.04} seconds')